In [75]:
markdown_text="""# Product Team Sync - May 15, 2023

## Attendees
- Sarah Chen (Product Lead)
- Mike Johnson (Engineering)
- Anna Smith (Design)
- David Park (QA)

## Agenda

### 1. Sprint Review
* Completed Features
  * User authentication flow
  * Dashboard redesign
  * Performance optimization
    * Reduced load time by 40%
    * Implemented caching solution
* Pending Items
  * Mobile responsive fixes
  * Beta testing feedback integration

### 2. Current Challenges
* Resource constraints in QA team
* Third-party API integration delays
* User feedback on new UI
  * Navigation confusion
  * Color contrast issues

### 3. Next Sprint Planning
* Priority Features
  * Payment gateway integration
  * User profile enhancement
  * Analytics dashboard
* Technical Debt
  * Code refactoring
  * Documentation updates

## Action Items
- [ ] @sarah: Finalize Q3 roadmap by Friday
- [ ] @mike: Schedule technical review for payment integration
- [ ] @anna: Share updated design system documentation
- [ ] @david: Prepare QA resource allocation proposal

## Next Steps
* Schedule individual team reviews
* Update sprint board
* Share meeting summary with stakeholders

## Notes
* Next sync scheduled for May 22, 2023
* Platform demo for stakeholders on May 25
* Remember to update JIRA tickets

---
Meeting recorded by: Sarah Chen
Duration: 45 minutes
"""

In [76]:
# installing the google client library for python
!pip install --upgrade google-api-python-client google-auth-httplib2 google-auth-oauthlib

In [77]:
import google.auth
SCOPES = ["https://www.googleapis.com/auth/documents"]
credentials, project = google.auth.default(scopes=SCOPES)


In [78]:
from google.colab import auth
auth.authenticate_user()


In [79]:
from googleapiclient.discovery import build
service = build("docs", "v1", credentials=credentials)

In [80]:
import re
from googleapiclient.errors import HttpError

def parse_markdown_to_requests(markdown_content):

    requests = []
    current_index = 1
    document_title = "Untitled Document"
    lines = markdown_content.strip().split("\n")

    for line in lines:
        raw_line = line.rstrip()
        formatted_text = ""
        paragraph_style = None

        if raw_line.startswith("# "):
            formatted_text = raw_line[2:].strip() + "\n"
            document_title = formatted_text.strip()
            paragraph_style = "HEADING_1"
        elif raw_line.startswith("## "):
            formatted_text = raw_line[3:].strip() + "\n"
            paragraph_style = "HEADING_2"
        elif raw_line.startswith("### "):
            formatted_text = raw_line[4:].strip() + "\n"
            paragraph_style = "HEADING_3"

        elif re.match(r'^\s*[-*]\s+', raw_line):
            indentation = len(raw_line) - len(raw_line.lstrip(" "))
            nesting_level = indentation // 2

            is_checkbox = bool(re.match(r'^\s*[-*]\s*\[[ xX]\]', raw_line))
            formatted_text = re.sub(r'^\s*[-*]\s*(\[[ xX]\])?\s*', '', raw_line).strip() + "\n"

            requests.append({
                "insertText": {
                    "location": {"index": current_index},
                    "text": formatted_text
                }
            })

            bullet_preset = "BULLET_CHECKBOX" if is_checkbox else "BULLET_DISC_CIRCLE_SQUARE"

            requests.append({
                "createParagraphBullets": {
                    "range": {
                        "startIndex": current_index,
                        "endIndex": current_index + len(formatted_text)
                    },
                    "bulletPreset": bullet_preset
                }
            })

            if nesting_level > 0:
                requests.append({
                    "updateParagraphStyle": {
                        "range": {"startIndex": current_index, "endIndex": current_index + len(formatted_text)},
                        "paragraphStyle": {
                            "indentStart": {"magnitude": 18 * nesting_level, "unit": "PT"},
                            "indentFirstLine": {"magnitude": 18 * nesting_level, "unit": "PT"}
                        },
                        "fields": "indentStart,indentFirstLine"
                    }
                })

            current_index += len(formatted_text)
            continue

        else:
            formatted_text = raw_line + "\n"

        requests.append({
            "insertText": {
                "location": {"index": current_index},
                "text": formatted_text
            }
        })

        if paragraph_style:
            end_idx = current_index + len(formatted_text)
            requests.append({
                "updateParagraphStyle": {
                    "range": {"startIndex": current_index, "endIndex": end_idx},
                    "paragraphStyle": {"namedStyleType": paragraph_style},
                    "fields": "namedStyleType"
                }
            })
            current_index = end_idx
        else:
            for match in re.finditer(r'(@\w+)', formatted_text):
                start, end = match.span()
                requests.append({
                    "updateTextStyle": {
                        "range": {
                            "startIndex": current_index + start,
                            "endIndex": current_index + end
                        },
                        "textStyle": {"bold": True},
                        "fields": "bold"
                    }
                })
            current_index += len(formatted_text)

    return document_title, requests


In [81]:
def create_and_format_doc(service, title, requests_list):

    if not service:
        print("Google Docs service is not available. Aborting.")
        return None

    try:
        print(f"Creating new Google Doc with title: '{title}'...")
        doc_body = {'title': title}
        doc = service.documents().create(body=doc_body).execute()
        doc_id = doc.get('documentId')
        print(f"Successfully created document with ID: {doc_id}")

        if requests_list:
            print("Applying formatting...")
            result = service.documents().batchUpdate(
                documentId=doc_id, body={'requests': requests_list}
            ).execute()
            print("Formatting applied successfully.")

        doc_url = f'https://docs.google.com/document/d/{doc_id}/edit'
        return doc_url

    except HttpError as error:
        print(f"An HTTP error occurred: {error}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None

In [82]:
doc_title, requests_list = parse_markdown_to_requests(markdown_text)
document_url = create_and_format_doc(service, doc_title, requests_list)
print(document_url)

Creating new Google Doc with title: 'Product Team Sync - May 15, 2023'...
Successfully created document with ID: 1YWeKY0GkW978HpeSoG06L8EIItgcVZ6Zf8guct1Jj18
Applying formatting...
Formatting applied successfully.
https://docs.google.com/document/d/1YWeKY0GkW978HpeSoG06L8EIItgcVZ6Zf8guct1Jj18/edit
